In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC 
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib   

In [ ]:
df = pd.read_csv('advanced_hand_data.csv')
print(f"[+] Dataset Shape: {df.shape}")
print(df.describe())

In [ ]:
X = df.drop(columns=['Target_Label'])
y = df['Target_Label']

unique_classes = np.unique(y)
all_target_names = {0: "FIST (0)", 1: "PEACE (1)", 2: "PALM (2)"}
active_target_names = [all_target_names[c] for c in unique_classes]

# 80-20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("[*] Tuning SVM hyperparameters using GridSearchCV...")
# probability=True is required for soft voting. 
# n_jobs=1 is explicitly set to prevent Windows Jupyter kernel crash bug.
param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'], 'kernel': ['rbf']}
grid_search = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=3, n_jobs=1)
grid_search.fit(X_train, y_train)
best_svm = grid_search.best_estimator_
print(f"[+] Best SVM Parameters: {grid_search.best_params_}")

print("[*] Training Random Forest and Gradient Boosting estimators...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

print("[*] Building Soft Voting Ensemble Classifier...")
ensemble_brain = VotingClassifier(
    estimators=[('tuned_svm', best_svm), ('random_forest', rf_model), ('gradient_boosting', gb_model)],
    voting='soft'
)
ensemble_brain.fit(X_train, y_train)

y_pred = ensemble_brain.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

print("\n================ PRODUCTION METRICS SUMMARY ================")
print(f"Final Ensemble Test Accuracy: {final_accuracy * 100:.2f}%\n")
print("Detailed Performance Matrix (Classification Report):")
print(classification_report(y_test, y_pred, labels=unique_classes, target_names=active_target_names))

joblib.dump(ensemble_brain, 'advanced_hand_model.pkl')
print("[+] Ensemble brain model saved as 'advanced_hand_model.pkl'!")